# Edge TinyML — Full Analysis Notebook
**Student:** Remmy Kipruto Tumo

Includes:
- Training
- Confusion Matrix
- Classification Report
- TFLite Latency Test
- SHAP Explainability Snippet


In [ ]:
!pip install -q tensorflow==2.12.0 scikit-learn shap seaborn matplotlib

In [ ]:
import tensorflow as tf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import shap, time, os
print('TF version:', tf.__version__)

In [ ]:
DATA_DIR = '/content/data'
IMG_SIZE = (160,160)
BATCH_SIZE = 32
train_ds = tf.keras.preprocessing.image_dataset_from_directory(DATA_DIR+'/train', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')
val_ds = tf.keras.preprocessing.image_dataset_from_directory(DATA_DIR+'/val', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')
test_ds = tf.keras.preprocessing.image_dataset_from_directory(DATA_DIR+'/test', image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')
class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)


In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE+(3,), include_top=False, weights='imagenet')
base_model.trainable = False
inputs = tf.keras.Input(shape=IMG_SIZE+(3,))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(train_ds, epochs=10, validation_data=val_ds)

In [ ]:
# Test set evaluation
loss, acc = model.evaluate(test_ds)
print('Test accuracy:', acc)

y_true=[]; y_pred=[]
for imgs, labels in test_ds:
    preds = model.predict(imgs)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.show()
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
model.save('model_recyclables.h5')

In [ ]:
def representative_data_gen():
    for images,labels in train_ds.take(50): yield [tf.cast(images,tf.float32).numpy()]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations=[tf.lite.Optimize.DEFAULT]
converter.representative_dataset=representative_data_gen
converter.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type=tf.uint8
converter.inference_output_type=tf.uint8
tflite_model = converter.convert()
open('model_recyclables_quant.tflite','wb').write(tflite_model)

In [ ]:
interpreter = tf.lite.Interpreter(model_path='model_recyclables_quant.tflite')
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

latencies=[]
for imgs,labels in test_ds.take(2):
    for img in imgs:
        img = tf.cast(tf.image.resize(img, IMG_SIZE), tf.uint8).numpy()[None,...]
        start=time.time(); interpreter.set_tensor(input_details[0]['index'], img)
        interpreter.invoke(); lat=(time.time()-start)*1000
        latencies.append(lat)

print('Mean latency:', np.mean(latencies), 'ms')

In [ ]:
try:
    bg=[]
    for imgs,labels in train_ds.take(1):
        for img in imgs: bg.append(img.numpy());
    bg=np.array(bg)
    explainer = shap.GradientExplainer((model.layers[0].input, model.layers[-1].output), bg)
    test_img = next(iter(test_ds.take(1)))[0][0].numpy()[None,...]
    sv = explainer.shap_values(test_img)
    print('SHAP computed.')
except Exception as e:
    print('SHAP unavailable in this environment:', e)